In [30]:
from openai import OpenAI
import requests
from minsearch import AppendableIndex
import json

In [31]:
openai_client = OpenAI()

In [32]:
def download_docs():
    docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
    docs_response = requests.get(docs_url)
    documents_raw = docs_response.json()

    documents = []

    for course in documents_raw:
        course_name = course['course']

        for doc in course['documents']:
            doc['course'] = course_name
            documents.append(doc)

    return documents

In [33]:
documents = download_docs()

In [34]:
index = AppendableIndex(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

index.fit(documents)

In [35]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=5,
    )

    return results

In [36]:
def make_call(call):
    f_name = call.name
    args = json.loads(call.arguments)

    if f_name == 'search':
        results = search(**args)
        results_json = json.dumps(results)
    else:
        raise ValueError(f'unknown function {f_name}')

    call_output = {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": results_json,
    }

    return call_output

In [37]:
# describe the search tool available for the agent
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [38]:
instructions = """
    You're a course teaching assistant.
    You're given a question from a course student and your task is to answer it.
    If you want to look up an answer, explain why before make the call
""".strip()

In [39]:
tools = [search_tool]

In [40]:
question = "I just discovered the course. Can I still join it?"

chat_messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question}
]

In [ ]:
while True:
    # prompt the LLM with the search tool
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=chat_messages,
        tools=tools
    )

    # append the LLM output to prompt context
    chat_messages.extend(response.output)
    has_function_call = False

    for item in response.output:
        # print the LLM message to the screen
        if item.type == 'message':
            print('Assistant:')
            print(item.content[0].text)
            print()
        
        # call the agent function and add output to context
        if item.type == 'function_call':
            call_output = make_call(item)
            chat_messages.append(call_output)
            has_function_call = True

    # exit when the agent no longer makes use of search tool
    if not has_function_call:
        break

Assistant:
To provide you with the most accurate information, I will check the course FAQ to see if there are any specific details about joining the course after it has already started.

Assistant:
Yes, you can still join the course! Even if you haven't registered, you're eligible to submit homework assignments. However, keep in mind that there will be deadlines for the final projects, so it's best not to procrastinate.

If you're aiming for a certificate, just note that you need to complete it with a "live" cohort, as certificates are not awarded for self-paced completion.

